In [ ]:
from pathlib import Path
import sys
import rasterio
from rasterio.merge import merge
from rasterio.enums import Resampling
from rasterio.vrt import WarpedVRT
from rasterio.windows import Window
import geopandas as gpd

In [ ]:
folders_2020 = [
    Path(r"C:\FAO_commodities_presence_absence\raw_ee\cocoa_2020"),
    Path(r"C:\FAO_commodities_presence_absence\raw_ee\coffee_2020"),
    Path(r"C:\FAO_commodities_presence_absence\raw_ee\palm_2020"),
    Path(r"C:\FAO_commodities_presence_absence\raw_ee\rubber_2020"),
    Path(r"C:\FAO_commodities_presence_absence\raw_ee\maize_2020"),
    Path(r"C:\FAO_commodities_presence_absence\raw_ee\timber_2020"),
]
folders_2025 = [
    Path(r"C:\FAO_commodities_presence_absence\raw_ee\cocoa_2025"),
    Path(r"C:\FAO_commodities_presence_absence\raw_ee\coffee_2025"),
    Path(r"C:\FAO_commodities_presence_absence\raw_ee\palm_2025"),
    Path(r"C:\FAO_commodities_presence_absence\raw_ee\rubber_2025"),
    Path(r"C:\FAO_commodities_presence_absence\raw_ee\maize_2025"),
    Path(r"C:\FAO_commodities_presence_absence\raw_ee\timber_2025"),
]
master_shapefile_path = Path(r"C:\Users\AFahrezi\Documents\GitHub\generic_rs_code\raster_mosaicking\AOI\FAO_RCP51_Country.shp") 
sea_gdf = gpd.read_file(master_shapefile_path)



In [ ]:
#config
folders = [
    Path(r"C:\Data Spasial\FAO_Binary_Commodities_FPDP\commodities_country_processed\2020_data_process\cocoa_2020\*_binary_*_clipped.tif"),
    Path(r"C:\Data Spasial\FAO_Binary_Commodities_FPDP\commodities_country_processed\2020_data_process\coffee_2020\*_binary_*_clipped.tif"),
    Path(r"C:\Data Spasial\FAO_Binary_Commodities_FPDP\commodities_country_processed\2020_data_process\palm_2020\*_binary_*_clipped.tif"),
    Path(r"C:\Data Spasial\FAO_Binary_Commodities_FPDP\commodities_country_processed\2020_data_process\rubber_2020\*_binary_*_clipped.tif"),
    Path(r"C:\Data Spasial\FAO_Binary_Commodities_FPDP\commodities_country_processed\2020_data_process\maize_2020\*_binary_*_clipped.tif"),
    #Path(r"C:\Data Spasial\FAO_Binary_Commodities_FPDP\commodities_country_processed\2020_data_process\rice_2020\*_binary_*_clipped.tif"),
    Path(r"C:\Data Spasial\FAO_Binary_Commodities_FPDP\commodities_country_processed\2020_data_process\timber_2020\*_binary_*_clipped.tif"),
]


OUTPUT_DIR = Path(r"C:\Data Spasial\FAO_Binary_Commodities_FPDP\mosaicked_all_countries\mosaic_2020_data")
OUTPUT_PATH = OUTPUT_DIR / "mosaic_2020_data.tif"
NODATA = 0                       # binary rasters: 0 = absence
DTYPE = "uint8"                  # binary maps are usually uint8
BLOCK = 1024                     # block size (px) for windowed read/write
RESAMPLING = Resampling.nearest  # nearest = correct choice for binary/categorical data
TARGET_CRS = "EPSG:4326"          # reproject every input to geographic coordinates

In [ ]:
def simple_merge(tif_files, out_path, nodata=NODATA):
    """
    Expands folders or wildcard paths, reprojects every raster to EPSG:4326,
    and merges them using disk-backed windows to limit memory usage.
    """
    raster_paths = []
    for item in tif_files:
        path = Path(item)
        path_text = str(path)
        if any(wildcard in path_text for wildcard in "*?["):
            matches = sorted(path.parent.glob(path.name))
        elif path.is_dir():
            matches = sorted(path.glob("*.tif"))
        else:
            matches = [path] if path.suffix.lower() == ".tif" else []
        raster_paths.extend(match for match in matches if match.is_file())

    if not raster_paths:
        raise FileNotFoundError("No TIFF files were found in the configured folders or patterns.")

    srcs = [rasterio.open(path) for path in raster_paths]
    warped_srcs = []
    try:
        target_res = next(
            src.res for src in srcs if src.crs and src.crs.to_string() == TARGET_CRS
        )
        merge_srcs = []
        for src in srcs:
            warped = WarpedVRT(
                src,
                crs=TARGET_CRS,
                resolution=target_res,
                nodata=nodata,
                resampling=RESAMPLING,
            )
            warped_srcs.append(warped)
            merge_srcs.append(warped)

        profile = srcs[0].profile.copy()
        profile.update(
            crs=TARGET_CRS,
            nodata=nodata,
            dtype=DTYPE,
            compress="lzw",
            predictor=2,
            tiled=True,
            blockxsize=BLOCK,
            blockysize=BLOCK,
            bigtiff="IF_SAFER",
        )
        out_path = Path(out_path)
        out_path.parent.mkdir(parents=True, exist_ok=True)
        merge(
            merge_srcs,
            nodata=nodata,
            dtype=DTYPE,
            resampling=RESAMPLING,
            mem_limit=512,
            dst_path=out_path,
            dst_kwds=profile,
        )
    finally:
        for src in warped_srcs:
            src.close()
        for src in srcs:
            src.close()

In [ ]:
mosaic_paths = []
for folder_pattern in folders:
    commodity_name = folder_pattern.parent.name.removesuffix("_2020")
    output_path = OUTPUT_DIR / f"{commodity_name}_2020_mosaic.tif"
    simple_merge([folder_pattern], output_path)
    mosaic_paths.append(output_path)

mosaic_paths

In [1]:
import mosaicking as mc
mc.main

<function mosaicking.main()>